# Hiligaynon NER: Deep Error Diagnostics & Confusion Matrix

This notebook generates granular confusion matrices to expose semantic tagging boundaries. It is structured to help manually inspect and isolate misclassifications, heavily centering on the `ORG` vs `GPE` constraint overlap and the linguistic challenges introduced by Taglish (code-switching).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import itertools

print('Diagnostic libraries loaded.')

In [ ]:
### 0. Load Converted Verified CoNLL Data

def load_conll_data(filepath):
    """Loads sentences and BIOES tags from a CoNLL-formatted file."""
    sentences = []
    sentence_labels = []
    
    if not os.path.exists(filepath):
        print(f"Warning: CoNLL file {filepath} not found.")
        return [], []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        words, labels = [], []
        for line in f:
            line = line.strip()
            if not line or line.startswith('-DOCSTART-'):
                if words:
                    sentences.append(words)
                    sentence_labels.append(labels)
                    words, labels = [], []
            else:
                splits = line.split()
                if len(splits) >= 2:
                    words.append(splits[0])
                    labels.append(splits[-1])
        if words:
            sentences.append(words)
            sentence_labels.append(labels)
    
    return sentences, sentence_labels

# Load test, train, and validation splits from converted_verified
converted_dir = os.path.join(os.getcwd(), '..', 'data', 'converted_verified')
print(f"Loading converted CoNLL files from: {converted_dir}\n")

test_sentences, test_labels = load_conll_data(os.path.join(converted_dir, 'dataset_test_final.conll'))
train_sentences, train_labels = load_conll_data(os.path.join(converted_dir, 'dataset_train_finak.conll'))
val_sentences, val_labels = load_conll_data(os.path.join(converted_dir, 'dataset_val_final.conll'))

print(f"Test set: {len(test_sentences)} sentences, {sum(len(s) for s in test_sentences)} tokens")
print(f"Train set: {len(train_sentences)} sentences, {sum(len(s) for s in train_sentences)} tokens")
print(f"Val set: {len(val_sentences)} sentences, {sum(len(s) for s in val_sentences)} tokens")
print(f"\nUnique labels in test set: {sorted(set(label for labels in test_labels for label in labels))}")

### 1. Generating the Heatmap

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title='NER Confusion Matrix', cmap='Reds'):
    """Plots a seaborn heatmap of the classification confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, 
                xticklabels=classes, yticklabels=classes, 
                cbar=False, linewidths=.5)
    
    plt.ylabel('True Labels', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Labels', fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Flatten test labels for analysis
y_true_flat = []
for labels in test_labels:
    y_true_flat.extend(labels)

unique_labels = sorted(set(y_true_flat))
print(f"Total test tokens: {len(y_true_flat)}")
print(f"Label distribution:")
for label in unique_labels:
    count = y_true_flat.count(label)
    pct = 100 * count / len(y_true_flat)
    print(f"  {label}: {count} ({pct:.1f}%)")

### 2. Hunting Down Misclassifications (ORG vs GPE & Taglish)
The function below is designed to filter out exact sentences where a specific true tag is systematically predicted as a designated false tag.

In [ ]:
def inspect_errors(sentences, true_labels, pred_labels, true_target='B-ORG', pred_target='B-GPE'):
    """
    Extracts contextual sentences where models confused true_target for pred_target.
    Used to identify systematic misclassifications (e.g., ORG vs GPE, Taglish code-mixing).
    """
    error_instances = []
    
    for sentence_idx, (words, t_tags, p_tags) in enumerate(zip(sentences, true_labels, pred_labels)):
        for word_idx, (word, t_tag, p_tag) in enumerate(zip(words, t_tags, p_tags)):
            if t_tag == true_target and p_tag == pred_target:
                context_start = max(0, word_idx - 2)
                context_end = min(len(words), word_idx + 3)
                context = " ".join(words[context_start:context_end])
                
                error_instances.append({
                    'sentence_idx': sentence_idx,
                    'word_idx': word_idx,
                    'word': word,
                    'context': context,
                    'true_tag': t_tag,
                    'pred_tag': p_tag
                })
    
    return error_instances

# Example: Get model predictions and analyze errors
# Once you run inference with your trained model, populate pred_test_labels
# Then analyze: 
# org_gpe_errors = inspect_errors(test_sentences, test_labels, pred_test_labels, 'B-ORG', 'B-GPE')
# for err in org_gpe_errors[:10]:
#     print(f"Word: '{err['word']}' | True: {err['true_tag']} -> Pred: {err['pred_tag']}")
#     print(f"Context: {err['context']}\n")

print("Error inspection functions ready.")
print("Note: Add model predictions to analyze systematic misclassifications.")